# DermArbiter — Paper-Ready Evaluation Notebook

**Owner:** Emre (Mahmut) — task E23 / #9 
**Inputs:** JSONL outputs from `dermarbiter.experiments.runner.BenchmarkRunner` and `dermarbiter.experiments.ablation.AblationRunner`. 
**Outputs:** Tables 1–5 + Figures 2/3/4/5 referenced in the Nature Medicine paper plan.

## Expected JSONL files
| Section | Path | Source script |
|---|---|---|
| Table 1: HAM10000 main | `results/ham10000.jsonl` | `python -m dermarbiter.experiments.runner --benchmark ham10000` |
| Table 2: Fitzpatrick17k fairness | `results/fitzpatrick17k.jsonl` | `python -m dermarbiter.experiments.runner --benchmark fitzpatrick17k` |
| Table 3: Agent LOO | `results/ablation_agent.jsonl` | `python -m dermarbiter.experiments.ablation --ablation-type agent` |
| Table 4: Tool LOO | `results/ablation_tool.jsonl` | `python -m dermarbiter.experiments.ablation --ablation-type tool` |
| Table 5: Debate rounds | `results/ablation_round.jsonl` | `python -m dermarbiter.experiments.ablation --ablation-type round` |

Until those JSONL files exist (waiting on GPU/API integration — Phase 2), all cells fall back to synthetic placeholders so the notebook executes end-to-end.

In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from dermarbiter.evaluation import (
    AblationAnalyzer,
    FairnessAnalyzer,
    MetricsCalculator,
)

RESULTS = Path("../results")
FIGS = Path("../results/figures")
FIGS.mkdir(parents=True, exist_ok=True)


def _load_or_synth(path: Path, synth_fn):
    """Return real JSONL rows if present; otherwise synthesised placeholders."""
    if path.exists():
        with open(path, "r", encoding="utf-8") as fh:
            return [json.loads(line) for line in fh if line.strip()], True
    return synth_fn(), False


print("Results directory:", RESULTS.resolve())

## Table 1 — HAM10000 main benchmark (DermArbiter vs. baselines)

Reports top-1 accuracy, balanced accuracy, macro-F1, AUROC, ECE, and Brier score on the 642-image DermAgent-aligned HAM10000 subset.

Published baseline numbers (DermAgent, MICCAI 2026) live in [`experiments/dermagent_baseline.md`](../experiments/dermagent_baseline.md).

In [ ]:
def _synth_ham():
    rng = np.random.default_rng(0)
    classes = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]
    rows = []
    for i in range(120):
        gt = classes[i % len(classes)]
        # 70% baseline accuracy in the placeholder
        pred = gt if rng.random() < 0.70 else classes[(i + 1) % len(classes)]
        rows.append({
            "case_id": f"HAM_{i:04d}",
            "predicted": pred,
            "ground_truth": gt,
            "final_diagnosis": [pred, classes[(i + 2) % 7]],
            "consensus_score": float(rng.uniform(0.55, 0.95)),
            "total_tokens": int(rng.integers(800, 2400)),
            "latency_ms": float(rng.uniform(800, 3500)),
            "num_debate_rounds": int(rng.integers(1, 4)),
            "total_tool_calls": int(rng.integers(3, 8)),
            "early_exit": bool(rng.random() < 0.4),
        })
    return rows

rows, real = _load_or_synth(RESULTS / "ham10000.jsonl", _synth_ham)
calc = MetricsCalculator(records=rows)
report = calc.compute_all()

src = "REAL results" if real else "SYNTHETIC placeholder (waiting on GPU benchmark)"
print(f"Source: {src} — n={report['n_cases']}")
table1 = {
    "Top-1 Acc": report["accuracy"],
    "Bal-Acc": report["balanced_accuracy"],
    "Macro-F1": report["macro_f1"],
    "AUROC": report["auroc"],
    "ECE": report["ece"],
    "Brier": report["brier_score"],
    "Latency p50 (ms)": report["latency_p50"],
    "Latency p95 (ms)": report["latency_p95"],
    "Throughput (cases/min)": report["throughput_cases_per_min"],
    "Cost/case (USD)": report["avg_cost_usd"],
}
for k, v in table1.items():
    print(f"  {k:30s} {v:.4f}")

### Figure 2 — Confusion matrix (HAM10000)

In [ ]:
cm = calc.confusion_matrix()
labels = sorted({r["ground_truth"] for r in rows})
mat = np.array([[cm.get(gt, {}).get(pred, 0) for pred in labels] for gt in labels], dtype=float)
row_sums = mat.sum(axis=1, keepdims=True)
mat_norm = np.divide(mat, row_sums, out=np.zeros_like(mat), where=row_sums > 0)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(mat_norm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(labels)), labels, rotation=45, ha="right")
ax.set_yticks(range(len(labels)), labels)
ax.set_xlabel("Predicted"); ax.set_ylabel("Ground truth")
ax.set_title("HAM10000 confusion matrix (row-normalised)" + ("" if real else " — placeholder"))
for i in range(len(labels)):
    for j in range(len(labels)):
        if mat[i, j]:
            ax.text(j, i, f"{int(mat[i, j])}", ha="center", va="center",
                    color="white" if mat_norm[i, j] > 0.5 else "black", fontsize=8)
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.savefig(FIGS / "fig2_confusion_ham10000.png", dpi=150)
plt.show()

## Table 2 — Fitzpatrick17k fairness (subgroup accuracy)

Per-Fitzpatrick-type accuracy + equalised odds + demographic parity gaps.

In [ ]:
def _synth_fitz():
    rng = np.random.default_rng(7)
    classes = ["benign", "malignant", "non_neoplastic"]
    types = ["I", "II", "III", "IV", "V", "VI"]
    rows = []
    for i in range(180):
        fp = types[i % len(types)]
        gt = classes[i % len(classes)]
        # Darker skin: lower baseline accuracy (the bias we want to surface)
        acc = {"I": 0.80, "II": 0.78, "III": 0.72, "IV": 0.66, "V": 0.55, "VI": 0.48}[fp]
        pred = gt if rng.random() < acc else classes[(i + 1) % len(classes)]
        rows.append({
            "case_id": f"F17K_{i:04d}",
            "predicted": pred,
            "ground_truth": gt,
            "consensus_score": float(rng.uniform(0.4, 0.95)),
            "fitzpatrick_type": fp,
        })
    return rows

rows_fp, real_fp = _load_or_synth(RESULTS / "fitzpatrick17k.jsonl", _synth_fitz)
fairness = FairnessAnalyzer(records=rows_fp, group_key="fitzpatrick_type")
fair_report = fairness.compute_all()

print("Source:", "REAL" if real_fp else "SYNTHETIC placeholder")
for grp in sorted(fair_report["group_names"]):
    n = fair_report["per_group_n"][grp]
    acc = fair_report["per_group_accuracy"][grp]
    print(f"  Type {grp:3s}  n={n:>3d}  acc={acc:.3f}")
print("\n  max accuracy gap        :", round(fair_report["accuracy_gap"]["max_delta"], 4))
dp = fair_report["demographic_parity"]
eo = fair_report["equalized_odds"]
print("  demographic parity gap  :", round(dp.get("max_pr_disparity", dp.get("demographic_parity_difference", 0.0)), 4))
print("  equalised odds gap      :", round(eo.get("max_tpr_disparity", eo.get("equalized_odds_difference", 0.0)), 4))

### Figure 3 — Fairness: per-Fitzpatrick accuracy with parity gap

In [ ]:
groups = sorted(fair_report["group_names"])
accs = [fair_report["per_group_accuracy"][g] for g in groups]
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(groups, accs, color="#3b7dd8")
ax.axhline(np.mean(accs), color="gray", linestyle="--", label=f"mean = {np.mean(accs):.2f}")
ax.set_xlabel("Fitzpatrick type")
ax.set_ylabel("Top-1 accuracy")
ax.set_ylim(0, 1)
ax.set_title("Fitzpatrick17k — per-subgroup accuracy" + ("" if real_fp else " (placeholder)"))
ax.legend()
for b, a in zip(bars, accs):
    ax.text(b.get_x() + b.get_width() / 2, a + 0.02, f"{a:.2f}",
            ha="center", fontsize=9)
plt.tight_layout()
plt.savefig(FIGS / "fig3_fairness_fitzpatrick.png", dpi=150)
plt.show()

## Tables 3-5 — Ablation studies

Uses `AblationAnalyzer` to compute Δacc + bootstrap 95% CI + paired p-value relative to the all-tools / all-agents baseline.

In [ ]:
def _synth_ablation(kind: str):
    rng = np.random.default_rng({"agent": 1, "tool": 2, "round": 3}[kind])
    rows = []
    base_acc = 0.72
    if kind == "agent":
        variants = [("baseline_all_agents", base_acc),
                    ("no_specialist", base_acc - 0.11),
                    ("no_generalist", base_acc - 0.06),
                    ("no_skeptic", base_acc - 0.04)]
    elif kind == "tool":
        variants = [("baseline_all_tools", base_acc),
                    ("no_panderm", base_acc - 0.08),
                    ("no_make", base_acc - 0.04),
                    ("no_dermogpt", base_acc - 0.05),
                    ("no_case_rag", base_acc - 0.03),
                    ("no_guideline_rag", base_acc - 0.02)]
    else:  # round
        variants = [("max_rounds_1", base_acc - 0.10),
                    ("max_rounds_2", base_acc - 0.04),
                    ("baseline_all_tools", base_acc),
                    ("max_rounds_5", base_acc + 0.01)]
    for vname, acc in variants:
        for i in range(40):
            gt = "mel"
            pred = gt if rng.random() < acc else "nv"
            rows.append({
                "variant": vname, "ablation_type": kind,
                "case_id": f"c{i:03d}",
                "predicted": pred, "ground_truth": gt,
                "latency_ms": float(rng.uniform(900, 2800) * (1.3 if kind == "round" and vname == "max_rounds_5" else 1.0)),
                "num_debate_rounds": {"max_rounds_1": 1, "max_rounds_2": 2, "max_rounds_5": 5}.get(vname, 3),
            })
    return rows

ablation_outputs = {}
for kind, label in [("agent", "Table 3 — Agent LOO"),
                    ("tool",  "Table 4 — Tool LOO"),
                    ("round", "Table 5 — Debate rounds")]:
    rows_ab, real_ab = _load_or_synth(RESULTS / f"ablation_{kind}.jsonl", lambda k=kind: _synth_ablation(k))
    analyzer = AblationAnalyzer(rows_ab, bootstrap_n=1000, seed=42)
    analyzer.compute()
    src = "REAL" if real_ab else "SYNTHETIC"
    print(f"\n### {label}  ({src})")
    print(analyzer.to_markdown())
    ablation_outputs[kind] = analyzer

### Figure 4 — Latency vs accuracy Pareto (efficiency tradeoff)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for kind, marker, color in [("tool", "o", "#1f77b4"),
                            ("agent", "s", "#d62728"),
                            ("round", "^", "#2ca02c")]:
    stats = ablation_outputs[kind].compute()
    first = True
    for s in stats:
        ax.scatter(s.mean_latency_ms, s.accuracy, marker=marker, color=color,
                   s=80, edgecolor="black", alpha=0.85,
                   label=f"{kind} ablation" if first else None)
        first = False
        ax.annotate(s.variant.replace("baseline_all_", "B:").replace("no_", "-"),
                    (s.mean_latency_ms, s.accuracy), fontsize=7,
                    xytext=(4, 4), textcoords="offset points")
ax.set_xlabel("Mean latency per case (ms)")
ax.set_ylabel("Top-1 accuracy")
ax.set_title("Efficiency Pareto — accuracy vs latency")
ax.legend(loc="lower right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIGS / "fig4_latency_accuracy_pareto.png", dpi=150)
plt.show()

### Figure 5 — Uncertainty calibration (reliability diagram)

Compares predicted confidence vs empirical accuracy in N equal-width bins. A perfectly-calibrated model would lie on y=x.

In [ ]:
n_bins = 10
edges = np.linspace(0.0, 1.0, n_bins + 1)
confs = np.array([float(r.get("consensus_score", 0.0)) for r in rows])
correct = np.array([
    1.0 if str(r.get("predicted", "")).lower() == str(r.get("ground_truth", "")).lower() else 0.0
    for r in rows
])

bin_acc, bin_conf = [], []
for i in range(n_bins):
    lo, hi = edges[i], edges[i + 1]
    mask = (confs > lo) & (confs <= hi)
    if mask.sum() > 0:
        bin_acc.append(correct[mask].mean())
        bin_conf.append(confs[mask].mean())

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Perfect calibration")
ax.plot(bin_conf, bin_acc, "o-", color="#d62728", label=f"DermArbiter (ECE={calc.ece():.3f})")
ax.set_xlabel("Predicted confidence (consensus score)")
ax.set_ylabel("Empirical accuracy")
ax.set_title("Reliability diagram" + ("" if real else " — placeholder"))
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.legend(loc="upper left")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIGS / "fig5_reliability.png", dpi=150)
plt.show()

## Export combined report

Writes a single JSON the paper team can paste numbers from.

In [ ]:
combined = {
    "ham10000": calc.compute_all(),
    "fitzpatrick17k": fair_report,
    "ablation_agent": ablation_outputs["agent"].to_dict(),
    "ablation_tool": ablation_outputs["tool"].to_dict(),
    "ablation_round": ablation_outputs["round"].to_dict(),
}
out_path = RESULTS / "evaluation_summary.json"
out_path.parent.mkdir(parents=True, exist_ok=True)
with open(out_path, "w", encoding="utf-8") as fh:
    json.dump(combined, fh, indent=2, default=str)
print("Wrote", out_path)